# Screen IDX 31 Juli dan 3–7 Agustus 2026

Notebook ini menjalankan dua jadwal forecast:

- origin **30 Juli 2026** → target **31 Juli 2026**;
- origin **31 Juli 2026** → target **3–7 Agustus 2026**.

Dua checkpoint yang digunakan:

1. Validated no-refit epoch 15.
2. Production refit epoch 4.

Setiap checkpoint memakai `best_weights.json` miliknya sendiri dari
`BackTest Results`. Kandidat awal adalah maksimum 100 saham dengan
prediksi close positif. Skor akhir adalah jumlah
`percentile_rank(feature) × weight`, lalu dipilih top 30 per model.

Bobot berasal dari backtest 1D full-timeframe. Bobot yang sama
diterapkan pada setiap horizon target 3–7 Agustus sesuai kebutuhan
screening ini.

Bobot positif menyukai nilai feature yang tinggi; bobot negatif
menekan nilai feature yang tinggi dan relatif menyukai nilai rendah.

## 1. Clone repository dan tarik dua checkpoint

In [ ]:
from pathlib import Path
import os, subprocess

LOCAL_REPO = Path.cwd().resolve()
if LOCAL_REPO.name == "BackTest Kronos Screener":
    LOCAL_REPO = LOCAL_REPO.parent
REPO = LOCAL_REPO if (LOCAL_REPO / "Kronos IDX FineTune").exists() else Path("/kaggle/working/ISTL")
if not REPO.exists():
    env = os.environ.copy()
    env["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "https://github.com/zzeiidann/ISTL.git", str(REPO)], check=True, env=env)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)

MODEL_CONFIGS = [
    {
        "slug": "validated_no_refit_e15",
        "label": "Validated no-refit epoch 15",
        "checkpoint": REPO / "Kronos IDX FineTune/results/2026-07-30/validated-no-refit-e15/kronos_base_idx_all/best_model",
        "weights": REPO / "BackTest Kronos Screener/BackTest Results/validated_no_refit_e15_1d/summary/best_weights.json",
    },
    {
        "slug": "production_refit_e4",
        "label": "Production refit epoch 4",
        "checkpoint": REPO / "Kronos IDX FineTune/results/2026-07-30/refit-run-e4/production_model",
        "weights": REPO / "BackTest Kronos Screener/BackTest Results/production_refit_e4_1d/summary/best_weights.json",
    },
]
includes = ",".join(str(config["checkpoint"].relative_to(REPO) / "model.safetensors") for config in MODEL_CONFIGS)
subprocess.run(["git", "-C", str(REPO), "lfs", "pull", f"--include={includes}"], check=True)
for config in MODEL_CONFIGS:
    assert (config["checkpoint"] / "model.safetensors").exists(), config["checkpoint"]
    assert config["weights"].exists(), config["weights"]
print("Repository:", REPO)

: 

## 2. Install runtime Kaggle P100/T4

In [ ]:
%pip install -q --upgrade torch==2.3.1 --index-url https://download.pytorch.org/whl/cu118
%pip install -q einops==0.8.1 huggingface_hub==0.33.1 safetensors==0.6.2 pyarrow optuna==4.4.0 tqdm yfinance

## 3. Siapkan data, runtime, dan arti feature

In [ ]:
import gc, importlib.util, json, sys
import numpy as np
import pandas as pd
import torch
from IPython.display import display

runner_path = REPO / "BackTest Kronos Screener/run_backtest.py"
spec = importlib.util.spec_from_file_location("kronos_backtest", runner_path)
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)

pattern_dir = REPO / "BackTest Pattern Screener"
sys.path.insert(0, str(pattern_dir))
from pattern_screener.config import PatternConfig
from pattern_screener.pattern_ranker import (
    build_pattern_snapshot, enrich_candidates, rank_enriched_candidates,
)

# Refresh actual OHLCV 31 Juli sebelum membentuk stock features.
subprocess.run(
    [
        sys.executable,
        str(REPO / "Kronos IDX FineTune/update_daily_parquet.py"),
        "--date", "2026-07-31",
    ],
    check=True,
)

device = runner.configure_runtime(42)
prices = runner.load_prices(REPO)
featured = runner.add_stock_features(prices)
pattern_config = PatternConfig(base_score_weight=0.65, pattern_weight=0.35)
pattern_snapshot = build_pattern_snapshot(prices, pattern_config)
available_dates = set(featured["date"])
JULY_30 = pd.Timestamp("2026-07-30")
JULY_31 = pd.Timestamp("2026-07-31")
AUGUST_TARGETS = list(pd.date_range("2026-08-03", "2026-08-07", freq="D"))
assert JULY_30 in available_dates, "Data 30 Juli 2026 belum tersedia."

assert JULY_31 in available_dates, "Unduhan yfinance 31 Juli tidak masuk ke parquet."

SCREEN_JOBS = [
    {
        "name": "2026-07-31",
        "origin": JULY_30,
        "target_dates": [JULY_31],
        "horizons": [1],
    },
    {
        "name": "2026-08-03_to_07",
        "origin": JULY_31,
        "target_dates": AUGUST_TARGETS,
        "horizons": [1, 2, 3, 4, 5],
    },
]

kronos_dir = REPO / "Kronos IDX FineTune/Kronos"
if not (kronos_dir / "model/kronos.py").exists():
    kronos_dir = Path("/kaggle/working/Kronos")
    if not kronos_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/shiyu-coder/Kronos.git", str(kronos_dir)], check=True)
    subprocess.run(["git", "-C", str(kronos_dir), "checkout", "67b630e67f6a18c9e9be918d9b4337c960db1e9a"], check=True)
sys.path.insert(0, str(kronos_dir))
from model import Kronos, KronosPredictor, KronosTokenizer

FEATURE_MEANINGS = {
    "pred_high_gain_mean": "Rata-rata estimasi kenaikan high dari 5 jalur forecast",
    "pred_high_gain_median": "Median estimasi kenaikan high",
    "pred_hit5_probability": "Proporsi jalur forecast yang memprediksi high +5%",
    "pred_close_gain_mean": "Rata-rata estimasi kenaikan close",
    "pred_close_up_probability": "Proporsi jalur yang memprediksi close naik",
    "pred_range_mean": "Estimasi lebar high-low harian",
    "pred_high_gain_dispersion": "Ketidakpastian estimasi kenaikan high",
    "pred_low_gain_mean": "Rata-rata estimasi posisi low terhadap previous close",
    "hit5_rate_20": "Frekuensi historis menyentuh +5% dalam 20 sesi",
    "hit5_rate_60": "Frekuensi historis menyentuh +5% dalam 60 sesi",
    "mom_5": "Momentum close 5 sesi",
    "mom_20": "Momentum close 20 sesi",
    "volatility_20": "Volatilitas return 20 sesi, annualized",
    "volume_ratio_5_60": "Volume rata-rata 5 sesi / median 60 sesi",
    "turnover_accel": "Akselerasi turnover 5 sesi terhadap median 60 sesi",
    "range_20": "Rentang high-low selama 20 sesi",
    "drawdown_60": "Posisi close terhadap high 60 sesi",
}

## 4. Tampilkan bobot dan arah screening setiap model

In [ ]:
weight_rows = []
for config in MODEL_CONFIGS:
    weights = json.loads(config["weights"].read_text())
    for feature, weight in weights.items():
        weight_rows.append({
            "model": config["label"],
            "feature": feature,
            "weight": weight,
            "arah_screen": "pilih nilai tinggi" if weight > 0 else "tekan nilai tinggi",
            "yang_diukur": FEATURE_MEANINGS[feature],
        })
weight_table = pd.DataFrame(weight_rows)
display(weight_table.sort_values(["model", "weight"], key=lambda s: s.abs() if s.name == "weight" else s, ascending=[True, False]))

## 5. Forecast dan screen top 30 per model

In [ ]:
OUTPUT_DIR = REPO / "BackTest Pattern Screener" / "future_screens" / "2026-08-03_to_07"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
tokenizer = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base").to(device).eval()
all_scored = []
all_selected = []

for config in MODEL_CONFIGS:
    print("Running", config["label"])
    model = Kronos.from_pretrained(str(config["checkpoint"])).to(device).eval()
    predictor = KronosPredictor(model, tokenizer, device=str(device), max_context=512)
    weights = json.loads(config["weights"].read_text())
    model_candidates = []
    model_selected = []
    for job in SCREEN_JOBS:
        forecast = runner.forecast_origin(
            predictor=predictor,
            featured=featured,
            origin=job["origin"],
            future_dates=job["target_dates"],
            horizons=job["horizons"],
            paths=5,
            batch_size=32,
            lookback=120,
            seed=42,
        )
        candidates = runner.prepare_candidate_panel(forecast, top_positive=100, select=30)
        candidates["secondary_score"] = runner.score_with_weights(candidates, weights)
        candidates["model"] = config["slug"]
        for feature in runner.SCORE_FEATURES:
            candidates[f"contribution_{feature}"] = candidates[f"r_{feature}"] * weights[feature]
        contribution_columns = [f"contribution_{feature}" for feature in runner.SCORE_FEATURES]
        candidates["top_support"] = candidates[contribution_columns].idxmax(axis=1).str.removeprefix("contribution_")
        candidates["top_penalty"] = candidates[contribution_columns].idxmin(axis=1).str.removeprefix("contribution_")

        enriched = enrich_candidates(candidates, pattern_snapshot)
        scored_candidates, selected = rank_enriched_candidates(
            enriched, pattern_config, select=30,
        )
        scored_candidates["score_percentile"] = scored_candidates.groupby("target_date")["final_ranking_score"].rank(pct=True, method="average")
        selected["score_percentile"] = selected.groupby("target_date")["final_ranking_score"].rank(pct=True, method="average")
        model_candidates.append(scored_candidates)
        model_selected.append(selected)

    model_candidates = pd.concat(model_candidates, ignore_index=True)
    model_selected = pd.concat(model_selected, ignore_index=True)
    model_candidates.to_csv(OUTPUT_DIR / f"{config['slug']}_all_candidates.csv", index=False)
    model_selected.to_csv(OUTPUT_DIR / f"{config['slug']}_top30_by_date.csv", index=False)
    all_scored.append(model_candidates)
    all_selected.append(model_selected)
    for target_date, daily in model_selected.groupby("target_date", sort=True):
        print(config["label"], pd.Timestamp(target_date).date())
        display(daily[["selected_rank", "ticker", "secondary_score", "net_pattern_score", "final_ranking_score", "top_bullish_pattern", "pattern_penalty", "pred_close_gain_mean", "pred_hit5_probability", "top_support", "top_penalty"]])

    del predictor, model
    gc.collect()
    torch.cuda.empty_cache()

## 6. Consensus dua model dan download output

In [ ]:
scored = pd.concat(all_scored, ignore_index=True)
selected = pd.concat(all_selected, ignore_index=True)
consensus = (
    scored.groupby(["target_date", "ticker"], as_index=False)
    .agg(models_available=("model", "nunique"), mean_score_percentile=("score_percentile", "mean"))
)
selected_counts = selected.groupby(["target_date", "ticker"])["model"].nunique().rename("models_selected")
consensus = consensus.merge(selected_counts, on=["target_date", "ticker"], how="left").fillna({"models_selected": 0})
consensus["models_selected"] = consensus["models_selected"].astype(int)
consensus = consensus.sort_values(
    ["target_date", "models_selected", "mean_score_percentile"],
    ascending=[True, False, False],
)
consensus["consensus_rank"] = consensus.groupby("target_date").cumcount() + 1
consensus = consensus[consensus["consensus_rank"].le(30)].copy()
consensus.to_csv(OUTPUT_DIR / "consensus_top30_by_date.csv", index=False)
weight_table.to_csv(OUTPUT_DIR / "weight_meanings.csv", index=False)
for target_date, daily in consensus.groupby("target_date", sort=True):
    print("Consensus", pd.Timestamp(target_date).date())
    display(daily[["consensus_rank", "ticker", "models_selected", "models_available", "mean_score_percentile"]])

import shutil
from IPython.display import Javascript
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print("Download:", archive)
if str(REPO).startswith("/kaggle/working"):
    display(Javascript(
        "const a=document.createElement('a');"
        + f"a.href='/files/kaggle/working/{Path(archive).name}';"
        + f"a.download='{Path(archive).name}';"
        + "document.body.appendChild(a);a.click();a.remove();"
    ))
else:
    print("Local archive ready:", archive)